# Python for Chemists — Day 7
## CV b-value analysis: is your electrode capacitive or battery-type? 🔋⚡

*Promised at the end of Day 6 — here it is!*

**What you will learn today**
1. Plot CV curves at many scan rates in one figure
2. Find the anodic and cathodic **peak currents automatically**
3. Calculate the **b-value** from log(i) vs log(v), and read what it means
4. Split your current into **capacitive** and **diffusion-controlled** parts at every potential (**Dunn method**)
5. Get a **% contribution table** and the shaded CV figure reviewers ask for
6. Bonus: specific capacity (C/g) and capacitance (F/g) from CV
7. Load **your own** CV files

**How to run it:** open this notebook in Google Colab → upload `CV_scan_rates.csv` (folder icon on the left) → *Runtime → Run all*.

> 📌 **The sample data is simulated** (a battery-type metal-oxide electrode in 3 M KOH, 0–0.55 V vs Hg/HgO, 2.0 mg active mass). Because I built it myself, I know the true answer, so you can check that the method works before you use your own data.

## The idea in simple words 🏪

Think of your electrode as a **shop**.

- **Capacitive charge** = customers served **at the front counter** (the surface). Service is instant. If twice as many people walk in per minute (faster scan), you serve twice as many. → current grows with **v**
- **Diffusion-controlled charge** = customers whose order must come **from the warehouse at the back** (inside the material). Ions have to travel. When you rush, they can't keep up. → current grows only with **√v**

Two simple equations turn this into numbers:

| Method | Equation | What it tells you |
|---|---|---|
| **b-value** | $i_p = a\,v^{\,b}$ → $\log i_p = \log a + b\,\log v$ | **b ≈ 0.5** → diffusion-controlled (battery-type) · **b ≈ 1** → capacitive · in between → mixed |
| **Dunn method** | $i(V) = k_1 v + k_2 v^{1/2}$ → $\dfrac{i(V)}{v^{1/2}} = k_1 v^{1/2} + k_2$ | $k_1 v$ = capacitive current, $k_2 v^{1/2}$ = diffusion current, at **every potential** |

The b-value is one number for the whole electrode. The Dunn method gives you the **percentage** of each type at each scan rate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 100, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# ---------- settings: change these for your own data ----------
DATA_FILE = "CV_scan_rates.csv"   # columns: scan_rate_mVs, potential_V, current_mA
MASS_mg = 2.0                     # active mass on the electrode (for the bonus section)
ANODIC_PEAK_WINDOW = (0.30, 0.53)    # V: where to look for the oxidation peak
CATHODIC_PEAK_WINDOW = (0.15, 0.40)  # V: where to look for the reduction peak
DUNN_SCAN_RATE = 5                # mV/s: which CV to show in the shaded figure

## Step 1 — Load the data and plot all CV curves

In [ ]:
df = pd.read_csv(DATA_FILE)
rates = sorted(int(r) for r in df["scan_rate_mVs"].unique())
print("Scan rates found (mV/s):", [int(r) for r in rates])
print(df.head())

fig, ax = plt.subplots(figsize=(6.5, 4.8))
colors = plt.cm.viridis(np.linspace(0.05, 0.9, len(rates)))
for r, c in zip(rates, colors):
    d = df[df["scan_rate_mVs"] == r]
    ax.plot(d["potential_V"], d["current_mA"], color=c, lw=1.6, label=f"{r} mV/s")
ax.set_xlabel("Potential (V vs Hg/HgO)")
ax.set_ylabel("Current (mA)")
ax.set_title("CV curves at different scan rates")
ax.axhline(0, color="grey", lw=0.6)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig("Day07_CV_curves.png", dpi=300)
plt.show()

## Step 2 — Split each CV into its two sweeps and find the peaks

A CV has an **anodic sweep** (potential going up, oxidation) and a **cathodic sweep** (potential going down, reduction). Python tells them apart by checking whether the potential is rising or falling.

In [ ]:
def split_sweeps(d):
    # returns (anodic, cathodic) DataFrames, each sorted by potential
    E = d["potential_V"].to_numpy()
    dE = np.gradient(E)
    anodic = d[dE > 0].sort_values("potential_V")
    cathodic = d[dE < 0].sort_values("potential_V")
    return anodic, cathodic

peaks = []
for r in rates:
    an, ca = split_sweeps(df[df["scan_rate_mVs"] == r])
    a = an[an["potential_V"].between(*ANODIC_PEAK_WINDOW)]
    c = ca[ca["potential_V"].between(*CATHODIC_PEAK_WINDOW)]
    ia = a.loc[a["current_mA"].idxmax()]
    ic = c.loc[c["current_mA"].idxmin()]
    peaks.append({"scan_rate_mVs": r,
                  "Epa_V": ia["potential_V"], "ipa_mA": ia["current_mA"],
                  "Epc_V": ic["potential_V"], "ipc_mA": ic["current_mA"]})

peaks = pd.DataFrame(peaks)
peaks["dEp_mV"] = (peaks["Epa_V"] - peaks["Epc_V"]) * 1000
peaks.round(3)

👀 **Look at `dEp_mV`:** the gap between the two peaks grows as you scan faster. That is normal (slow kinetics + resistance). If it grows a lot, your fastest scan rates are less reliable — keep that in mind for the Dunn method.

## Step 3 — The b-value

Plot **log(peak current)** against **log(scan rate)**. The slope of the straight line is **b**.

In [ ]:
logv = np.log10(peaks["scan_rate_mVs"])

def b_fit(i_peak):
    y = np.log10(np.abs(i_peak))
    b, loga = np.polyfit(logv, y, 1)
    r2 = 1 - np.sum((y - (b * logv + loga))**2) / np.sum((y - y.mean())**2)
    return b, loga, r2

def meaning(b):
    if b < 0.6:  return "mostly diffusion-controlled (battery-type)"
    if b > 0.9:  return "mostly capacitive (surface-controlled)"
    return "mixed: both diffusion and capacitive storage"

b_a, la, r2a = b_fit(peaks["ipa_mA"])
b_c, lc, r2c = b_fit(peaks["ipc_mA"])
print(f"Anodic peak:   b = {b_a:.2f}  (R² = {r2a:.4f}) → {meaning(b_a)}")
print(f"Cathodic peak: b = {b_c:.2f}  (R² = {r2c:.4f}) → {meaning(b_c)}")

fig, ax = plt.subplots(figsize=(5.5, 4.3))
xx = np.linspace(logv.min() - 0.05, logv.max() + 0.05, 50)
ax.plot(logv, np.log10(np.abs(peaks["ipa_mA"])), "o", color="#c0392b", label=f"anodic, b = {b_a:.2f}")
ax.plot(xx, b_a * xx + la, "-", color="#c0392b", lw=1.2)
ax.plot(logv, np.log10(np.abs(peaks["ipc_mA"])), "s", color="#2471a3", label=f"cathodic, b = {b_c:.2f}")
ax.plot(xx, b_c * xx + lc, "-", color="#2471a3", lw=1.2)
ax.set_xlabel("log (scan rate / mV s$^{-1}$)")
ax.set_ylabel("log (|peak current| / mA)")
ax.set_title("b-value from peak currents")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("Day07_b_value.png", dpi=300)
plt.show()

## Step 4 — Dunn method: capacitive vs diffusion at every potential

For **each potential** (on each sweep), we take the current at all scan rates and fit a straight line:

$$\frac{i(V)}{v^{1/2}} = k_1\,v^{1/2} + k_2$$

Slope = **k₁**, intercept = **k₂**. Then the capacitive current at any scan rate is simply **k₁·v**.

In [ ]:
grid = np.linspace(df["potential_V"].min() + 0.01, df["potential_V"].max() - 0.01, 250)
v = np.array(rates) / 1000.0          # V/s
sqv = np.sqrt(v)

def currents_on_grid(sweep):
    # matrix (n_rates x n_grid) of current on a common potential grid
    out = []
    for r in rates:
        an, ca = split_sweeps(df[df["scan_rate_mVs"] == r])
        s = an if sweep == "anodic" else ca
        out.append(np.interp(grid, s["potential_V"], s["current_mA"]))
    return np.array(out)

dunn = {}
for sweep in ("anodic", "cathodic"):
    I = currents_on_grid(sweep)                         # mA
    k1, k2 = np.polyfit(sqv, I / sqv[:, None], 1)       # one fit per potential (vectorised)
    dunn[sweep] = {"I": I, "k1": k1, "k2": k2}

print("Fitted k1 and k2 at", len(grid), "potentials on each sweep ✅")

In [ ]:
rows = []
for j, r in enumerate(rates):
    cap = tot = 0.0
    for sweep in ("anodic", "cathodic"):
        i_tot = dunn[sweep]["I"][j]
        i_cap = dunn[sweep]["k1"] * v[j]
        cap += np.abs(i_cap).sum()
        tot += np.abs(i_tot).sum()
    rows.append({"scan_rate_mVs": r, "capacitive_%": 100 * cap / tot})

contrib = pd.DataFrame(rows)
contrib["diffusion_%"] = 100 - contrib["capacitive_%"]
contrib.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

# (a) shaded CV at one scan rate
j = rates.index(DUNN_SCAN_RATE)
ax = axes[0]
for sweep in ("anodic", "cathodic"):
    i_tot = dunn[sweep]["I"][j]
    i_cap = dunn[sweep]["k1"] * v[j]
    i_cap = np.where(np.abs(i_cap) > np.abs(i_tot), i_tot, i_cap)   # cannot be bigger than the total
    ax.plot(grid, i_tot, color="black", lw=1.4, label="measured CV" if sweep == "anodic" else None)
    ax.fill_between(grid, 0, i_cap, color="#5dade2", alpha=0.75,
                    label="capacitive" if sweep == "anodic" else None)
ax.axhline(0, color="grey", lw=0.6)
share = contrib.loc[j, "capacitive_%"]
ax.set_title(f"{DUNN_SCAN_RATE} mV/s: capacitive = {share:.0f} %")
ax.set_xlabel("Potential (V vs Hg/HgO)")
ax.set_ylabel("Current (mA)")
ax.legend(frameon=False, loc="upper left")

# (b) contribution bars at every scan rate
ax = axes[1]
x = np.arange(len(rates))
ax.bar(x, contrib["capacitive_%"], color="#5dade2", label="capacitive")
ax.bar(x, contrib["diffusion_%"], bottom=contrib["capacitive_%"], color="#f5b041", label="diffusion-controlled")
for xi, c in zip(x, contrib["capacitive_%"]):
    ax.text(xi, c / 2, f"{c:.0f}%", ha="center", va="center", fontsize=9)
ax.set_xticks(x, [str(r) for r in rates])
ax.set_xlabel("Scan rate (mV/s)")
ax.set_ylabel("Contribution (%)")
ax.set_ylim(0, 100)
ax.set_title("Charge storage contribution")
ax.legend(frameon=False, loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig("Day07_Dunn_contribution.png", dpi=300)
plt.show()

✅ **Check against the truth:** the sample data was built with capacitive shares of **33, 41, 49, 54, 60 and 68 %** (5 → 100 mV/s). The Dunn method finds **30 → 66 %**: the same trend, about 2–4 points lower. The small gap comes from the peaks moving a little with scan rate (see honest check 2 below) — a useful reminder that this method gives a good estimate, not an exact number.

**How to read it:** at slow scans, ions have time to reach the "warehouse", so diffusion dominates. At fast scans, only the "front counter" keeps up, so the capacitive share rises. This trend is what you expect to see.

## Bonus — Specific capacity and capacitance from CV

For one sweep, the charge is the area under the curve divided by the scan rate:

$$Q\,(\mathrm{C\,g^{-1}}) = \frac{\int |i|\,dV}{m\,v} \qquad C_s\,(\mathrm{F\,g^{-1}}) = \frac{Q}{\Delta V}$$

We use the **cathodic (discharge) sweep**. For a battery-type material like this one, reviewers increasingly want **C/g or mAh/g**, not only F/g.

In [ ]:
from scipy.integrate import trapezoid   # works on old and new numpy versions

m_g = MASS_mg / 1000
caps = []
for r in rates:
    _, ca = split_sweeps(df[df["scan_rate_mVs"] == r])
    area_mA_V = trapezoid(np.abs(ca["current_mA"]), ca["potential_V"])   # mA·V
    Q = (area_mA_V / 1000) / (m_g * r / 1000)                             # C/g
    dV = ca["potential_V"].max() - ca["potential_V"].min()
    caps.append({"scan_rate_mVs": r, "Q_C_per_g": Q, "Q_mAh_per_g": Q / 3.6, "Cs_F_per_g": Q / dV})

caps = pd.DataFrame(caps)
caps.round(1)

## ⚠️ Honest checks before you put this in a paper

1. **Use the scan rates where your peaks are clear.** If a peak moves outside your potential window at high scan rates, leave those rates out of the b-value fit.
2. **The Dunn method assumes the peak does not move.** Real peaks shift with scan rate. If you get capacitive % above 100 or negative k₁ values, your fastest scans are too distorted — drop them.
3. **"Capacitive" here = double-layer + fast surface redox together.** This method cannot separate the two.
4. **Always report the scan-rate range you used** — the numbers change with it.
5. **A high capacitive share is not automatically "better".** It depends on whether you want power (fast) or energy (more charge).

## Use your own data 🧪

Most instruments save **one file per scan rate**. Export the cycle you want (usually the **2nd or 3rd** cycle, once the curve is stable) as CSV or TXT, then use the helper below. It builds a file with the same three columns used above.

In [ ]:
def load_my_cv(files, potential_col, current_col, current_unit="A", sep=","):
    # files: dict {scan_rate_mVs: "filename"}
    # current_unit: "A", "mA" or "uA" (what your instrument saves)
    factor = {"A": 1000, "mA": 1, "uA": 0.001}[current_unit]
    parts = []
    for rate, fname in files.items():
        d = pd.read_csv(fname, sep=sep)
        parts.append(pd.DataFrame({"scan_rate_mVs": rate,
                                   "potential_V": d[potential_col],
                                   "current_mA": d[current_col] * factor}))
    out = pd.concat(parts, ignore_index=True)
    out.to_csv("my_CV_scan_rates.csv", index=False)
    print("Saved my_CV_scan_rates.csv → set DATA_FILE to this name and run all cells again")
    return out

# Example (remove the # and change the names):
# my = load_my_cv({5: "cv_5.csv", 10: "cv_10.csv", 20: "cv_20.csv", 50: "cv_50.csv"},
#                 potential_col="WE(1).Potential (V)", current_col="WE(1).Current (A)", current_unit="A")

In [ ]:
# Save all result tables in one go
peaks.to_csv("Day07_peak_currents.csv", index=False)
contrib.to_csv("Day07_Dunn_contributions.csv", index=False)
caps.to_csv("Day07_CV_capacity.csv", index=False)
print("Saved: Day07_peak_currents.csv, Day07_Dunn_contributions.csv, Day07_CV_capacity.csv")
print("Figures: Day07_CV_curves.png, Day07_b_value.png, Day07_Dunn_contribution.png")

---
**Python for Chemists** — one practical skill every day. Day 1: XRD · Day 2: Scherrer · Day 3: Tauc · Day 4: Kinetics · Day 5: GCD · Day 6: EIS · **Day 7: CV b-value + Dunn** ✅

Questions or ideas for the next lesson? Comment on the LinkedIn post — I read every one.